# 01-02 활성화 함수는 공간을 접어요

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 선형 레이어 두 개는 선형 레이어 하나와 같아요

In [ ]:
layer1 = nn.Linear(2, 3)
layer2 = nn.Linear(3, 2)

x = torch.rand(5, 2)  # (5, 2)
y_two = layer2(layer1(x))  # (5, 2)

# 두 레이어를 하나로 합친 가중치와 편향
W = layer2.weight @ layer1.weight  # (2, 3) @ (3, 2) = (2, 2)
b = layer2.weight @ layer1.bias + layer2.bias  # (2,)
y_one = x @ W.T + b  # (5, 2)

print(torch.allclose(y_two, y_one, atol=1e-6))

## 2. ReLU: 음수를 0으로 바꾸기

In [ ]:
t = torch.linspace(-3, 3, 100)
plt.figure(figsize=(4, 3))
plt.plot(t, torch.relu(t), color="tab:orange")
plt.axhline(0, color="k", lw=0.8)
plt.axvline(0, color="k", lw=0.8)
plt.title("ReLU(x) = max(0, x)")
plt.show()

In [ ]:
print(torch.relu(torch.tensor([-2.0, -0.5, 0.0, 1.5, 3.0])))

## 3. ReLU를 2차원 점에 적용하기

In [ ]:
xs = torch.linspace(-2, 2, 9)
grid = torch.cartesian_prod(xs, xs)  # (81, 2)

W = torch.tensor([[1.0, 0.8], [-0.4, 1.2]])  # (2, 2), 01-01과 같은 가중치
b = torch.tensor([1.0, 0.5])  # (2,)

h = grid @ W.T + b  # (81, 2), 선형 레이어
r = torch.relu(h)  # (81, 2), 활성화 함수

# 점마다 원래 어느 사분면에 있었는지 색으로 표시해요
quad = (h[:, 0] > 0).long() * 2 + (h[:, 1] > 0).long()
colors = ["tab:red", "tab:green", "tab:purple", "tab:blue"]
print("사분면별 점 개수:", torch.bincount(quad).tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
for ax, pts, title in [(axes[0], grid, "input x"), (axes[1], h, "Wx + b"), (axes[2], r, "ReLU(Wx + b)")]:
    ax.scatter(pts[:, 0], pts[:, 1], c=[colors[q] for q in quad], s=18, edgecolor="k", lw=0.3)
    ax.axhline(0, color="k", lw=0.8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlim(-3.5, 4.5)
    ax.set_ylim(-3.5, 4.5)
    ax.set_aspect("equal")
    ax.set_title(title)
plt.show()

## 4. 꺾인 선을 더해 곡선 만들기

In [ ]:
torch.manual_seed(0)
x = torch.linspace(-3, 3, 200).unsqueeze(1)  # (200, 1)
target = torch.sin(x)  # (200, 1)

def bent_line_model(n_hidden):
    # Linear(1 -> n) -> ReLU -> Linear(n -> 1)
    return nn.Sequential(nn.Linear(1, n_hidden), nn.ReLU(), nn.Linear(n_hidden, 1))

results = {}
for n in [1, 3, 30]:
    torch.manual_seed(0)
    model = bent_line_model(n)
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    for step in range(2000):
        loss = ((model(x) - target) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    results[n] = model(x).detach()
    print(f"꺾인 선 {n:2d}개: 손실 {loss.item():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (n, pred) in zip(axes, results.items()):
    ax.plot(x, target, color="gray", lw=3, alpha=0.5, label="sin(x)")
    ax.plot(x, pred, color="tab:orange", label="model")
    ax.set_title(f"hidden = {n}")
    ax.legend()
plt.show()